# New-session restore for QE tutorials on Colab

Run this notebook at the start of each **new Colab session** (after a disconnect,
runtime reset, or on a different day). It mounts your Google Drive and restores the
QE environment in about 1 minute, so that the bootstrap cell in each tutorial notebook
runs in seconds instead of a minute.

> [!NOTE]
> **Prerequisite:** `qe_environment_setup.ipynb` must have been run at least once
> to save `qe_env.tar.gz` to `My Drive/qe_tutorial_colab/` on your Google Drive.

After this notebook shows all ✓, open any tutorial notebook and run its cells from
the beginning — the bootstrap cell will detect the environment is already restored
and skip the slow extraction step.

In [ ]:
from google.colab import drive
import sys, subprocess
from pathlib import Path

drive.mount('/content/drive')

DRIVE_DIR  = Path('/content/drive/MyDrive/qe_tutorial_colab')
QE_ENV_DIR = Path('/content/qe_env')
QE_BIN     = QE_ENV_DIR / 'bin'

if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))

if not QE_ENV_DIR.exists():
    print('Restoring QE environment (~1 min)...')
    subprocess.run(
        ['tar', '-xzf', str(DRIVE_DIR / 'qe_env.tar.gz'), '-C', '/content'],
        check=True,
    )
    print('Done.')
else:
    print('QE env already present — skipping extraction.')

In [ ]:
import importlib

ok = True

print('QE executables:')
for exe in ['pw.x', 'bands.x', 'dos.x', 'projwfc.x']:
    p = QE_BIN / exe
    if p.exists():
        print(f'  ✓  {exe}')
    else:
        print(f'  ✗  {exe}  — not found at {p}')
        ok = False

print('\nPseudopotentials:')
pseudo_dir = DRIVE_DIR / 'pseudo'
for ps in ['Mg.upf', 'O.upf']:
    p = pseudo_dir / ps
    if p.exists():
        print(f'  ✓  {ps}')
    else:
        print(f'  ✗  {ps}  — not found at {p}')
        ok = False

print('\nPython modules:')
for mod in ['numpy', 'matplotlib', 'scipy', 'ase',
            'pw_input', 'convergence_runner', 'eos_tools', 'elastic_tools', 'bandstructure_tools']:
    try:
        importlib.import_module(mod)
        print(f'  ✓  {mod}')
    except ImportError as e:
        print(f'  ✗  {mod}  — {e}')
        ok = False

print()
if ok:
    print('All checks passed — open a tutorial notebook and run from the top.')
else:
    print('Some checks failed — review the output above before continuing.')